**import packages**


In [ ]:

%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold

## read dataset + split train test per flow

In [ ]:
ddos_dataset_df = pd.read_csv("ddos_dataset_first_version_clean.csv", low_memory = False)
X = ddos_dataset_df.drop(columns=['label'])
y = ddos_dataset_df[['label']]


In [ ]:
# split train test per flow
# TODO replace naming to fit per flow 


X_train, X_test, y_train, y_test = train_test_split(
    X, # X
    y, # y
    stratify = y, # stratify the dataset based on class labels
    train_size = 0.7, # percentage of training set
    random_state = 15 
)


#TODO fix and change according if pca is done and thnk about maybe i want to normalize some of them
cols_to_exclude = ['flow_id', 'source_ip', 'destination_ip', 'timestamp', 'simillarhttp'] 

# normalize only selected columns
all_features = X_train.columns.tolist()
cols_to_scale = [col for col in all_features if col not in cols_to_exclude]
scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])




In [ ]:
#check stratified 
import pandas as pd

# 1. Calculate the normalized value counts (percentages) for both sets
# We use normalize=True to get fractions (0.10) instead of raw counts (100)
train_dist = y_train.value_counts(normalize=True).sort_index()
test_dist = y_test.value_counts(normalize=True).sort_index()

# 2. Combine them into a single DataFrame for easy comparison
comparison_df = pd.DataFrame({
    'Train_Ratio': train_dist,
    'Test_Ratio': test_dist
})

# 3. Calculate the difference to see the error margin
comparison_df['Diff'] = (comparison_df['Train_Ratio'] - comparison_df['Test_Ratio']).abs()

# 4. Display the table (Formatted as percentages)
print("Stratification Check (Class Distribution):")
print(comparison_df.style.format("{:.2%}"))

# 5. Optional: Alert on big discrepancies
# If any class differs by more than 5% (0.05), it might be problematic
threshold = 0.05
bad_splits = comparison_df[comparison_df['Diff'] > threshold]

if not bad_splits.empty:
    print(f"\n⚠️ WARNING: The following classes are poorly stratified (> {threshold:.0%} diff):")
    print(bad_splits.index.tolist())
else:
    print("\n✅ Stratification looks good! (All differences < 5%)")

## show that this is not reasnoble to split per group per flow

In [ ]:
#TODO validate not sure this is what requested
# should the split be some src ip in train different src ip in dest or litteraly use groupby and then look only at ip level 
"""
# split train test per flow + normalization
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=14)
groups = ddos_dataset_df[' Source IP']
# get splited dataset, iterate one iteration of generaotr
for train_index, test_index in gss.split(X, y, groups=groups):
    X_train_ip, X_test_ip = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train_ip, y_test_ip = y.iloc[train_index].copy(), y.iloc[test_index].copy()

#TODO fix and change according if pca is done and thnk about maybe i want to normalize some of them
cols_to_exclude = ['Flow ID', ' Source IP', ' Destination IP', 'Protocol', ' Timestamp', 'SimillarHTTP','Source Port','Destination Port'] 

# normalize only selected columns
all_features = X_train_ip.columns.tolist()
cols_to_scale = [col for col in all_features if col not in cols_to_exclude]
ip_scaler = StandardScaler()
X_train_ip[cols_to_scale] = ip_scaler.fit_transform(X_train_ip[cols_to_scale])
X_test_ip[cols_to_scale] = ip_scaler.transform(X_test_ip[cols_to_scale])
"""

In [ ]:
#unsuccessful attemp to use stratifed group
"""
groups = ddos_dataset_df['source_ip']
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=16)
for train_index, test_index in sgkf.split(X, y, groups=groups):
    X_train_ip, X_test_ip = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train_ip, y_test_ip = y.iloc[train_index].copy(), y.iloc[test_index].copy()
    break

#TODO fix and change according if pca is done and thnk about maybe i want to normalize some of them
cols_to_exclude = ['flow_id', 'source_ip', 'destination_ip', 'protocol', 'timestamp', 'simillarhttp','source_port','destination_port'] 

# normalize only selected columns
all_features = X_train_ip.columns.tolist()
cols_to_scale = [col for col in all_features if col not in cols_to_exclude]
ip_scaler = StandardScaler()
X_train_ip[cols_to_scale] = ip_scaler.fit_transform(X_train_ip[cols_to_scale])
X_test_ip[cols_to_scale] = ip_scaler.transform(X_test_ip[cols_to_scale])
"""

In [ ]:
print(ddos_dataset_df['source_ip'].value_counts(normalize=True).sort_values(ascending=False))



In [ ]:
# 1. Aggregate the data (Counting flows per Source IP)
flow_counts = ddos_dataset_df['source_ip'].value_counts().sort_values(ascending=False)

# 2. Plot the top 20 IPs for readability
top_n = 20
plot_data = flow_counts.head(top_n)

# 3. Create the Bar Plot
plt.figure(figsize=(12, 6))
sns.barplot(x=plot_data.index, y=plot_data.values, palette="viridis")
plt.title(f'Number of Flows per Source IP (Top {top_n})', fontsize=16)
plt.xlabel('Source IP Address', fontsize=12)
plt.ylabel('Number of Flows (Count)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10) 
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

In [ ]:
ip_label_counts = pd.crosstab(ddos_dataset_df['source_ip'], ddos_dataset_df['label'])
print("without specific ip")
print(ip_label_counts.drop("172.16.0.5").sum())
print("specific ip")
print(ip_label_counts.loc["172.16.0.5"])

## create + train models

In [ ]:
# remove unnecessary columns for training 
cols_to_drop = ['flow_id', 'source_ip', 'destination_ip', 'timestamp','simillarhttp']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

In [ ]:


log_reg = LogisticRegression()
log_reg.fit(X_train, y_train.values.ravel())
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train.values.ravel())
dtc = DecisionTreeClassifier()
dtc.fit(X_train, y_train.values.ravel())
gnb = GaussianNB()
gnb.fit(X_train, y_train.values.ravel())



## show results

In [ ]:

#present results
y_train_log_reg_pred = log_reg.predict(X_train)
y_test_log_reg_pred = log_reg.predict(X_test)
print("log_reg")
print(classification_report(y_train, y_train_log_reg_pred)) 
print(classification_report(y_test, y_test_log_reg_pred)) 

y_train_rfc_pred = rfc.predict(X_train)
y_test_rfc_pred = rfc.predict(X_test)
print("rfc")
print(classification_report(y_train, y_train_rfc_pred)) 
print(classification_report(y_test, y_test_rfc_pred)) 


y_train_dtc_pred = dtc.predict(X_train)
y_test_dtc_pred = dtc.predict(X_test)
print("dtc")
#TODO figure out why conversion is needed
print(classification_report(y_train, y_train_dtc_pred.astype(int))) 
print(classification_report(y_test, y_test_dtc_pred.astype(int))) 


y_train_gnb_pred = gnb.predict(X_train)
y_test_gnb_pred = gnb.predict(X_test)
print("gnb")
print(classification_report(y_train, y_train_gnb_pred)) 
print(classification_report(y_test, y_test_gnb_pred)) 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import accuracy_score

# 1. Calculate the exact accuracy scores for all models
# We assume the predictions (y_train_log_reg_pred, etc.) are already in memory from your previous step

scores = [
    {
        'Model': 'Logistic Regression',
        'Train': accuracy_score(y_train, y_train_log_reg_pred),
        'Test': accuracy_score(y_test, y_test_log_reg_pred)
    },
    {
        'Model': 'Random Forest',
        'Train': accuracy_score(y_train, y_train_rfc_pred),
        'Test': accuracy_score(y_test, y_test_rfc_pred)
    },
    {
        'Model': 'Decision Tree',
        'Train': accuracy_score(y_train, y_train_dtc_pred.astype(int)), # Handling your dtype fix
        'Test': accuracy_score(y_test, y_test_dtc_pred.astype(int))
    },
    {
        'Model': 'Gaussian NB',
        'Train': accuracy_score(y_train, y_train_gnb_pred),
        'Test': accuracy_score(y_test, y_test_gnb_pred)
    }
]

# 2. Convert to DataFrame and "Melt" for easier plotting with Seaborn
df_scores = pd.DataFrame(scores)
df_melted = df_scores.melt(id_vars="Model", var_name="Set", value_name="Accuracy")

# 3. Create the Grouped Bar Chart
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Create the bar plot
ax = sns.barplot(
    data=df_melted, 
    x="Model", 
    y="Accuracy", 
    hue="Set", 
    palette="viridis"
)

# 4. Add the exact numbers on top of the bars (Annotation)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=10)

# 5. Customize Layout
plt.title('Model Comparison: Train vs Test Accuracy', fontsize=16)
plt.ylim(0, 1.1) # Set Y-axis slightly above 1 to make room for labels
plt.ylabel('Accuracy Score', fontsize=12)
plt.xlabel('Classifier', fontsize=12)
plt.legend(title='Dataset', loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()

plt.show()
#TODO write that  logistic regression is not as good because not all data can be sepereted linearly

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Organize your predictions into a list
# Note: I used .astype(int) to ensure they are clean integers based on your comments
predictions_list = [
    ("Logistic Regression", y_test_log_reg_pred),
    ("Random Forest", y_test_rfc_pred),
    ("Decision Tree", y_test_dtc_pred.astype(int)), 
    ("Gaussian NB", y_test_gnb_pred.astype(int))
]

# 2. Create the subplot framework (1 Row, 4 Columns)
# figsize=(24, 5) makes it wide enough so they don't look squashed
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# 3. Loop through the axes and predictions together
for ax, (model_name, y_pred) in zip(axes, predictions_list):
    
    # Calculate the confusion matrix for this specific model
    cm = confusion_matrix(y_test, y_pred)
    
    # Plot heatmap
    # fmt='d' ensures numbers are integers (no scientific notation)
    # cbar=False turns off the color bar to save space (colors are relative anyway)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    
    # Set labels
    ax.set_title(model_name, fontsize=14)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

# Adjust layout to prevent overlapping
plt.tight_layout()
plt.show()

## Logistic regression parameters fine tune

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
{
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1','l2'],
    "solver": ['saga']
},
{
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    "solver": ['lbfgs']
},
{
    'penalty': [None],
    "solver": ['lbfgs','saga']
}
             ]

#TODO increase max_iter
log_reg = LogisticRegression(max_iter=30)

grid = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    cv=2,#TODO return and increse
    scoring="accuracy",
    return_train_score=False,
    verbose=3,
    n_jobs=-1
)

grid.fit(X_train, y_train.values.ravel())


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Extract results to a DataFrame for easy plotting
results_df = pd.DataFrame(grid.cv_results_)

# Clean up: Convert "NaN" parameters (which appear when a param isn't in a specific grid) to string "None"
results_df['param_C'] = results_df['param_C'].fillna('None')
results_df['param_penalty'] = results_df['param_penalty'].fillna('None')
results_df['param_solver'] = results_df['param_solver'].fillna('None')

# ==========================================
# GRAPH 1: Grid 1 (Saga Solver - L1 vs L2)
# ==========================================
# Filter for 'saga' and ensure we only get rows where penalty is NOT 'None'
grid1_df = results_df[
    (results_df['param_solver'] == 'saga') & 
    (results_df['param_penalty'] != 'None')
].copy()

# Create Matrix (Pivot Table)
pivot_saga = grid1_df.pivot(index='param_C', columns='param_penalty', values='mean_test_score')

plt.figure(figsize=(6, 5))
sns.heatmap(pivot_saga, annot=True, cmap='viridis', fmt='.4f')
plt.title('Grid 1: Solver "Saga" (L1 vs L2)')
plt.ylabel('C (Regularization)')
plt.xlabel('Penalty')
plt.tight_layout()
plt.show()

# ==========================================
# GRAPH 2: Grid 2 (LBFGS Solver - L2 Only)
# ==========================================
# Filter for 'lbfgs' and 'l2' penalty
grid2_df = results_df[
    (results_df['param_solver'] == 'lbfgs') & 
    (results_df['param_penalty'] == 'l2')
].copy()

# Create Matrix (Pivot Table) - This will be a single column
pivot_lbfgs = grid2_df.pivot(index='param_C', columns='param_penalty', values='mean_test_score')

plt.figure(figsize=(4, 5))
sns.heatmap(pivot_lbfgs, annot=True, cmap='Blues', fmt='.4f', cbar=False)
plt.title('Grid 2: Solver "LBFGS" (L2 Only)')
plt.ylabel('C (Regularization)')
plt.xlabel('Penalty')
plt.tight_layout()
plt.show()

# ==========================================
# GRAPH 3: Grid 3 (No Penalty - Solver Comparison)
# ==========================================
# Filter for rows where penalty is 'None'
grid3_df = results_df[results_df['param_penalty'] == 'None'].copy()

plt.figure(figsize=(8, 5))
# A Bar chart is best here because we are just comparing categories (Solvers)
ax = sns.barplot(x='param_solver', y='mean_test_score', data=grid3_df, palette='Set2')

# Add the exact score on top of the bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=3)

plt.title('Grid 3: No Penalty (Solver Comparison)')
plt.ylabel('Accuracy Score')
plt.xlabel('Solver')
plt.ylim(0, 1.1) # Set Y limits to make room for labels
plt.tight_layout()
plt.show()

# ==========================================
# SAVE BEST MODEL
# ==========================================
best_log_reg = grid.best_estimator_

print("-" * 30)
print(f"Best Model saved to variable: best_log_reg")
print(f"Best Accuracy: {grid.best_score_:.4f}")
print(f"Best Parameters: {grid.best_params_}")
print("-" * 30)

## Random forest parameters fine tune

In [ ]:
from sklearn.model_selection import GridSearchCV


"""
param_grid = {
    # 1. Number of trees (Stability)
    'n_estimators': [100, 200, 500],
    
    # 2. Max depth (Complexity constraint)
    'max_depth': [None, 10, 20, 30],
    
    # 3. Features per split (Diversity)
    'max_features': ['sqrt', 'log2'],
    
    # 4. Minimum sizes (Regularization/Smoothing)
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4]
}
"""
param_grid = {
    # 1. Number of trees (Stability)
    #'n_estimators': [100, 200, 500],
    'n_estimators': [100, 200, 500],

    # 2. Max depth (Complexity constraint)
    'max_depth': [10, 20, 30],
    
    # 3. Features per split (Diversity)
    #'max_features': ['sqrt', 'log2'],
    
    # 4. Minimum sizes (Regularization/Smoothing)
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4]
}

#TODO increase max_iter
rf = RandomForestClassifier(random_state=15)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=2,#TODO return and increse
    scoring="accuracy",
    return_train_score=False,
    verbose=3,
    n_jobs=-1
)

grid.fit(X_train, y_train.values.ravel())


In [ ]:
best_rf = grid.best_estimator_

print("-" * 30)
print(f"Best Model saved to variable: best_rf")
print(f"Best Accuracy: {grid.best_score_:.4f}")
print(f"Best Parameters: {grid.best_params_}")
print("-" * 30)

In [ ]:
# 3. Process Results into a DataFrame
results = pd.DataFrame(grid.cv_results_)
#cols = ['param_n_estimators', 'param_max_depth', 'param_max_features', 
#        'param_min_samples_split', 'param_min_samples_leaf', 'mean_test_score']
cols = ['param_n_estimators', 'param_max_depth',  
        'param_min_samples_split', 'param_min_samples_leaf', 'mean_test_score']
df = results[cols].copy()
#df.columns = ['n_estimators', 'max_depth', 'max_features', 
#              'min_samples_split', 'min_samples_leaf', 'score']
df.columns = ['n_estimators', 'max_depth',  
              'min_samples_split', 'min_samples_leaf', 'score']
df['max_depth'] = df['max_depth'].fillna('None')

# 4. Plot Box Plots for ALL 5 Parameters
fig, axes = plt.subplots(1, 5, figsize=(24, 5), sharey=True)

#params_to_plot = ['n_estimators', 'max_depth', 'max_features', 
#                  'min_samples_split', 'min_samples_leaf']
params_to_plot = ['n_estimators', 'max_depth',  
                  'min_samples_split', 'min_samples_leaf']

for i, param in enumerate(params_to_plot):
    sns.boxplot(x=param, y='score', data=df, ax=axes[i])
    axes[i].set_title(f"Impact of {param}")
    axes[i].set_xlabel(param)
    if i > 0:
        axes[i].set_ylabel("") # Hide y-label for inner plots to save space

plt.tight_layout()
plt.show()

## Decision Tree parameters fine tune